# Explore WordNet with NLTK

This notebook helps you inspect **WordNet** entries for any English word (with a focus on nouns).

## What is WordNet?
WordNet is a large lexical database of English where words are grouped into sets of cognitive synonyms called **synsets**.

## What is a synset?
A **synset** is a specific sense of a word (for example, one sense of `bank` means a financial institution, another means land by a river).

## What this notebook displays
For each synset found for a word, this notebook shows:
- Synset name
- Part of speech
- Definition (gloss)
- Example sentences
- Lemmas (synonyms)
- Hypernyms and hyponyms
- Holonyms and meronyms (member/part/substance)
- Attributes, entailments, and `similar_tos` (when available)

It also includes optional advanced helpers to inspect hypernym paths and compare two words side by side.

In [1]:
# Setup cell: ensure NLTK and WordNet resources are available.
# This cell is robust: it installs/downloads only when needed.

import sys
import subprocess
import importlib

def _ensure_package(package_name):
    """Install a Python package with pip if it is not already importable."""
    try:
        importlib.import_module(package_name)
        print(f"Package '{package_name}' is already installed.")
    except ImportError:
        print(f"Package '{package_name}' not found. Installing...")
        subprocess.check_call([sys.executable, "-m", "pip", "install", package_name])
        print(f"Installed '{package_name}'.")

# Ensure NLTK itself is installed.
_ensure_package("nltk")

import nltk
from nltk.corpus import wordnet as wn

def _ensure_nltk_resource(resource_path, download_name):
    """Download an NLTK corpus/model if missing."""
    try:
        nltk.data.find(resource_path)
        print(f"NLTK resource '{download_name}' is available.")
    except LookupError:
        print(f"NLTK resource '{download_name}' missing. Downloading...")
        nltk.download(download_name)

# WordNet corpus and multilingual index used by many WordNet installs.
_ensure_nltk_resource("corpora/wordnet", "wordnet")
_ensure_nltk_resource("corpora/omw-1.4", "omw-1.4")

print("Setup complete. Ready to inspect WordNet.")

Package 'nltk' not found. Installing...
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 1.6/1.6 MB 77.9 MB/s  0:00:00
Installed 'nltk'.
NLTK resource 'wordnet' missing. Downloading...


[nltk_data] Downloading package wordnet to /home/xiaoyue/nltk_data...


NLTK resource 'omw-1.4' missing. Downloading...


[nltk_data] Downloading package omw-1.4 to /home/xiaoyue/nltk_data...


Setup complete. Ready to inspect WordNet.


In [2]:
# Core inspection utilities.
# Main entry point: inspect_wordnet(word, pos=None)

from typing import List, Optional

# Human-friendly labels and WordNet POS mappings.
POS_MAP = {
    None: None,
    "noun": wn.NOUN,
    "verb": wn.VERB,
    "adjective": wn.ADJ,
    "adverb": wn.ADV,
}

POS_LABEL = {
    "n": "noun",
    "v": "verb",
    "a": "adjective",
    "s": "adjective satellite",
    "r": "adverb",
}

def _clean_list(values: List[str], empty_text: str = "(none)") -> str:
    """Return a readable comma-separated string, even for empty lists."""
    if not values:
        return empty_text
    return ", ".join(values)

def _synset_items(synsets) -> List[str]:
    """Format related synsets as 'name — definition'."""
    items = []
    for s in synsets:
        try:
            items.append(f"{s.name()} — {s.definition()}")
        except Exception:
            # Fallback if any individual synset is malformed.
            items.append(str(s))
    return items

def _print_section(title: str, values: List[str]):
    """Consistent section rendering for notebook readability."""
    print(f"  {title}:")
    if not values:
        print("    (none)")
        return
    for item in values:
        print(f"    - {item}")

def inspect_wordnet(word: str, pos: Optional[str] = None):
    """
    Inspect WordNet synsets for a given word.

    Parameters
    ----------
    word : str
        Input word to inspect.
    pos : str | None
        Optional POS filter: noun, verb, adjective, adverb, or None.
    """
    if not isinstance(word, str) or not word.strip():
        print("Please provide a non-empty word string.")
        return

    normalized_word = word.strip()
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None

    if normalized_pos not in POS_MAP:
        valid = ["noun", "verb", "adjective", "adverb", "None"]
        print(f"Invalid pos='{pos}'. Valid options: {', '.join(valid)}")
        return

    wn_pos = POS_MAP[normalized_pos]

    # Retrieve synsets with optional POS filtering.
    synsets = wn.synsets(normalized_word, pos=wn_pos) if wn_pos else wn.synsets(normalized_word)

    print("=" * 90)
    print(f"WordNet inspection for word: '{normalized_word}'")
    print(f"POS filter: {normalized_pos if normalized_pos else 'None (all parts of speech)'}")
    print(f"Synsets found: {len(synsets)}")
    print("=" * 90)

    # Robust handling when no synsets exist.
    if not synsets:
        print("No synsets found for this word with the selected POS filter.")
        return

    for idx, syn in enumerate(synsets, start=1):
        print(f"\n[{idx}] {syn.name()}")
        print("-" * 90)

        # Basic synset fields.
        pos_label = POS_LABEL.get(syn.pos(), syn.pos())
        print(f"  Synset name : {syn.name()}")
        print(f"  POS         : {pos_label}")
        print(f"  Definition  : {syn.definition() if syn.definition() else '(none)'}")

        examples = syn.examples() or []
        _print_section("Example sentences", examples)

        lemmas = [lemma.name().replace('_', ' ') for lemma in syn.lemmas()]
        _print_section("Lemma names (synonyms)", lemmas)

        # Relation sections required by the notebook specification.
        _print_section("Hypernyms", _synset_items(syn.hypernyms()))
        _print_section("Hyponyms", _synset_items(syn.hyponyms()))

        _print_section("Member holonyms", _synset_items(syn.member_holonyms()))
        _print_section("Part holonyms", _synset_items(syn.part_holonyms()))
        _print_section("Substance holonyms", _synset_items(syn.substance_holonyms()))

        _print_section("Member meronyms", _synset_items(syn.member_meronyms()))
        _print_section("Part meronyms", _synset_items(syn.part_meronyms()))
        _print_section("Substance meronyms", _synset_items(syn.substance_meronyms()))

        _print_section("Attributes", _synset_items(syn.attributes()))
        _print_section("Entailments", _synset_items(syn.entailments()))

        # similar_tos is mostly meaningful for adjective-like synsets.
        similar_tos = syn.similar_tos() if hasattr(syn, "similar_tos") else []
        _print_section("Similar_tos", _synset_items(similar_tos))

        print("-" * 90)

    print("\nInspection complete.")

## Optional Advanced Helpers

This section includes:
1. A helper to print hypernym path tree(s) for a selected synset.
2. A helper to compare two words by listing their synsets side by side.

In [3]:
# Advanced helper functions.

import textwrap

def print_hypernym_paths_for_word(word: str, pos: Optional[str] = None, synset_index: int = 0):
    """
    Print all hypernym paths for one selected synset of a word.

    Parameters
    ----------
    word : str
        Target word.
    pos : str | None
        Optional POS filter: noun, verb, adjective, adverb, or None.
    synset_index : int
        Zero-based index of the synset to inspect.
    """
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None
    if normalized_pos not in POS_MAP:
        print(f"Invalid pos='{pos}'.")
        return

    wn_pos = POS_MAP[normalized_pos]
    synsets = wn.synsets(word, pos=wn_pos) if wn_pos else wn.synsets(word)

    if not synsets:
        print(f"No synsets found for '{word}'.")
        return

    if synset_index < 0 or synset_index >= len(synsets):
        print(f"synset_index out of range. Choose 0 to {len(synsets)-1}.")
        return

    target = synsets[synset_index]
    paths = target.hypernym_paths() or []

    print("=" * 90)
    print(f"Hypernym paths for: {target.name()} — {target.definition()}")
    print(f"Total paths: {len(paths)}")
    print("=" * 90)

    if not paths:
        print("No hypernym paths available for this synset.")
        return

    for i, path in enumerate(paths, start=1):
        print(f"\nPath {i}:")
        for depth, node in enumerate(path):
            indent = "  " * depth
            print(f"{indent}- {node.name()} ({POS_LABEL.get(node.pos(), node.pos())})")

def compare_words_synsets(word1: str, word2: str, pos: Optional[str] = None, max_synsets: int = 8):
    """
    Compare synsets of two words side by side in plain text.
    """
    normalized_pos = pos.lower().strip() if isinstance(pos, str) else None
    if normalized_pos not in POS_MAP:
        print(f"Invalid pos='{pos}'.")
        return

    wn_pos = POS_MAP[normalized_pos]
    synsets1 = wn.synsets(word1, pos=wn_pos) if wn_pos else wn.synsets(word1)
    synsets2 = wn.synsets(word2, pos=wn_pos) if wn_pos else wn.synsets(word2)

    print("=" * 120)
    print(f"Synset comparison: '{word1}' vs '{word2}' | POS: {normalized_pos if normalized_pos else 'all'}")
    print(f"Counts: {len(synsets1)} vs {len(synsets2)}")
    print("=" * 120)

    if not synsets1 and not synsets2:
        print("No synsets found for either word.")
        return

    width = 58
    rows = max(min(len(synsets1), max_synsets), min(len(synsets2), max_synsets), 1)

    def _syn_line(s):
        if s is None:
            return "(none)"
        return f"{s.name()} — {s.definition()}"

    header_left = f"{word1} synsets"
    header_right = f"{word2} synsets"
    print(f"{header_left:<{width}} | {header_right:<{width}}")
    print("-" * width + "-+-" + "-" * width)

    for i in range(rows):
        s1 = synsets1[i] if i < len(synsets1) and i < max_synsets else None
        s2 = synsets2[i] if i < len(synsets2) and i < max_synsets else None

        left = textwrap.shorten(_syn_line(s1), width=width, placeholder="...")
        right = textwrap.shorten(_syn_line(s2), width=width, placeholder="...")
        print(f"{left:<{width}} | {right:<{width}}")

    if len(synsets1) > max_synsets or len(synsets2) > max_synsets:
        print("\n(Comparison truncated by max_synsets.)")

## Demo Cells

In [4]:
# Demo: director
inspect_wordnet("director")

WordNet inspection for word: 'director'
POS filter: None (all parts of speech)
Synsets found: 5

[1] director.n.01
------------------------------------------------------------------------------------------
  Synset name : director.n.01
  POS         : noun
  Definition  : someone who controls resources and expenditures
  Example sentences:
    (none)
  Lemma names (synonyms):
    - director
    - manager
    - managing director
  Hypernyms:
    - administrator.n.01 — someone who administers a business
  Hyponyms:
    - bank_manager.n.01 — manager of a branch office of a bank
    - district_manager.n.01 — a manager who supervises the sales activity for a district
    - manageress.n.01 — a woman manager
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Similar_tos:
    (none)
------------------------------

In [ ]:
# Demo: career
inspect_wordnet("career")

In [ ]:
# Demo: bank
inspect_wordnet("bank")

In [5]:
# Demo: apple
inspect_wordnet("apple")

WordNet inspection for word: 'apple'
POS filter: None (all parts of speech)
Synsets found: 2

[1] apple.n.01
------------------------------------------------------------------------------------------
  Synset name : apple.n.01
  POS         : noun
  Definition  : fruit with red or yellow or green skin and sweet to tart crisp whitish flesh
  Example sentences:
    (none)
  Lemma names (synonyms):
    - apple
  Hypernyms:
    - edible_fruit.n.01 — edible reproductive body of a seed plant especially one having sweet flesh
    - pome.n.01 — a fleshy fruit (apple or pear or related fruits) having seed chambers and an outer fleshy part
  Hyponyms:
    - eating_apple.n.01 — an apple used primarily for eating raw without cooking
    - crab_apple.n.03 — small sour apple; suitable for preserving
    - cooking_apple.n.01 — an apple used primarily in cooking for pies and applesauce etc
  Member holonyms:
    (none)
  Part holonyms:
    - apple.n.02 — native Eurasian tree widely cultivated in many 

In [6]:
# Noun-only inspection example (requirement 8).
inspect_wordnet("bank", pos="noun")

WordNet inspection for word: 'bank'
POS filter: noun
Synsets found: 10

[1] bank.n.01
------------------------------------------------------------------------------------------
  Synset name : bank.n.01
  POS         : noun
  Definition  : sloping land (especially the slope beside a body of water)
  Example sentences:
    - they pulled the canoe up on the bank
    - he sat on the bank of the river and watched the currents
  Lemma names (synonyms):
    - bank
  Hypernyms:
    - slope.n.01 — an elevated geological formation
  Hyponyms:
    - waterside.n.01 — land bordering a body of water
    - riverbank.n.01 — the bank of a river
  Member holonyms:
    (none)
  Part holonyms:
    (none)
  Substance holonyms:
    (none)
  Member meronyms:
    (none)
  Part meronyms:
    (none)
  Substance meronyms:
    (none)
  Attributes:
    (none)
  Entailments:
    (none)
  Similar_tos:
    (none)
------------------------------------------------------------------------------------------

[2] deposito

In [ ]:
# Advanced demo 1: hypernym path tree for the first noun sense of 'apple'.
print_hypernym_paths_for_word("apple", pos="noun", synset_index=0)

In [ ]:
# Advanced demo 2: side-by-side comparison.
compare_words_synsets("bank", "career", pos=None, max_synsets=6)